# Lab 18 — MITRE ATT&CK Threat Mapping with Python

*AI Cybersecurity Masterclass · Lab 18 · AI60secLab*

**Companion video:** [▶ MITRE ATT&CK: The Hacker Playbook, Explained](https://www.youtube.com/watch?v=aLlwzBgRYJg&list=PLG4l6Ssi9A00)

**Research question:** How can machine-readable MITRE ATT&CK data be transformed into an analyzable threat-technique dataset?

This notebook downloads the official Enterprise ATT&CK STIX 2.1 bundle at runtime. If network access is unavailable, place `enterprise-attack.json` in `data/raw/`.

## Security foundation: tactics, techniques, and the pyramid of pain

MITRE ATT&CK is the shared language defenders use for attacker behavior:
- a **tactic** is the attacker's goal at a stage (initial access, persistence, exfiltration);
- a **technique** is how they achieve it (phishing, scheduled tasks);
- a **procedure** is the specific way a particular group does it.

The **pyramid of pain** explains why behavior matters. Blocking a file hash or an IP address costs an attacker almost nothing, because they change it in minutes. Detecting their **techniques** forces them to change how they operate, which is expensive. Mapping detections to ATT&CK shows which behaviors you can see and where you are blind.

In [1]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

Project root: /home/claude/run/exec
Data folder : /home/claude/run/exec/data/raw


In [2]:
from src.mitre_data import (
    ATTACK_ENTERPRISE_URL,
    load_remote_or_local,
    technique_catalog,
    object_catalog,
    relationship_table,
)

local_fallback = DATA_DIR / "enterprise-attack.json"

attack_bundle = load_remote_or_local(
    ATTACK_ENTERPRISE_URL,
    local_path=local_fallback,
)

techniques = technique_catalog(
    attack_bundle,
    id_prefix="T",
)
groups = object_catalog(
    attack_bundle,
    "intrusion-set",
)
relationships = relationship_table(
    attack_bundle,
)

print("Active techniques/sub-techniques:", len(techniques))
print("Active groups:", len(groups))
print("Active relationships:", len(relationships))

display(techniques.head())

Active techniques/sub-techniques: 706
Active groups: 178
Active relationships: 21822


,stix_id,external_id,name,tactics,description,modified
0,attack-pattern--0042a9f5-f053-4769-b3ef-9ad018...,T1055.011,Extra Window Memory Injection,"privilege-escalation, stealth",Adversaries may inject malicious code into a p...,2026-11-04T03:09:58.175Z
1,attack-pattern--005a06c6-14bf-4118-afa0-ebcd8a...,T1053.005,Scheduled Task,"execution, persistence, privilege-escalation",Adversaries may abuse the Windows Task Schedul...,2026-05-12T15:12:00.618Z
2,attack-pattern--005cc321-08ce-4d17-b1ea-cb5275...,T1205.002,Socket Filters,"command-and-control, persistence, stealth",Adversaries may attach filters to a network so...,2026-05-12T15:12:00.619Z
3,attack-pattern--00f90846-cbd1-4fc5-9233-df5c2b...,T1560.001,Archive via Utility,collection,Adversaries may use utilities to compress and/...,2026-05-12T15:12:00.619Z
4,attack-pattern--01327cde-66c4-4123-bf34-5f258d...,T1021.005,VNC,lateral-movement,Adversaries may use [Valid Accounts](https://a...,2025-10-24T17:48:19.567Z


In [3]:
tactic_rows = techniques.copy()
tactic_rows["tactic"] = tactic_rows["tactics"].str.split(", ")
tactic_rows = tactic_rows.explode("tactic")
tactic_rows = tactic_rows[
    tactic_rows["tactic"].notna()
    & (tactic_rows["tactic"] != "")
]

tactic_counts = (
    tactic_rows.groupby("tactic", as_index=False)
    .agg(technique_count=("external_id", "nunique"))
    .sort_values("technique_count", ascending=False)
)

display(tactic_counts)

,tactic,technique_count
14,stealth,151
10,persistence,113
11,privilege-escalation,96
2,credential-access,68
5,execution,66
3,defense-impairment,57
13,resource-development,51
4,discovery,49
12,reconnaissance,46
1,command-and-control,45


In [4]:
query = "credential"

mask = (
    techniques["name"].str.contains(
        query, case=False, na=False
    )
    | techniques["description"].str.contains(
        query, case=False, na=False
    )
)

display(
    techniques.loc[
        mask,
        ["external_id", "name", "tactics"]
    ].head(30)
)

,external_id,name,tactics
4,T1021.005,VNC,lateral-movement
9,T1037,Boot or Logon Initialization Scripts,"persistence, privilege-escalation"
10,T1557,Adversary-in-the-Middle,"collection, credential-access"
11,T1033,System Owner/User Discovery,discovery
12,T1583,Acquire Infrastructure,resource-development
18,T1556.003,Pluggable Authentication Modules,"credential-access, defense-impairment, persist..."
22,T1056.001,Keylogging,"collection, credential-access"
25,T1110.001,Password Guessing,credential-access
27,T1597.002,Purchase Technical Data,reconnaissance
28,T1003,OS Credential Dumping,credential-access


## Research task

Choose one tactic and document:
- the techniques mapped to it;
- which behaviors your available telemetry could observe;
- which behaviors require endpoint, identity, cloud, or application telemetry.

**Important:** ATT&CK organizes observed adversary behavior. A network-flow classifier does not automatically produce ATT&CK attribution.

## From lab to SOC

Teams use ATT&CK for coverage maps (which techniques do our detections cover?), for red-team planning, and for reporting incidents in a shared vocabulary. A flow-based alert is a *hint* toward a technique, never proof of one. Analysts confirm with more evidence.

**Check yourself**
1. Why is detecting a technique more valuable than blocking an IP address?
2. Which ATT&CK tactics could flow data plausibly hint at, and which are nearly invisible to it?
3. Why does ATT&CK change between releases, and what does that mean for hardcoded technique IDs?